# Stage 15 — Native AlignTune: independent gold rescoring + blinded multilingual human review

**Purpose:** Improve research validity before Lexsi submission. This is CPU-only analysis of the exact original private Stage 13 48-case paired results ZIP. It **does not train, run new model inference, access Hugging Face or publish anything**.

Original source ZIP SHA256: `e0319c83fbb87ef8570ca0c42caf68bea4a4fef1f8ae320bbe11e9a7c1761c68`.

First we recompute all 96 saved answers against the frozen reference dataset; then, with owner opt-in, create a **private model-blinded CSV** for actual independent reviewers and a **separate OWNER-ONLY mapping file**. Never upload the secret mapping to public GitHub/HF or give it to reviewers. Human semantic correctness is NOT inferred.


## 1 — Fetch current project source on CPU

A GPU is **not** required. Do not rerun training notebooks 11 or 13. Reuse the original Stage 13 results ZIP, not the model adapter ZIP.


In [ ]:
from pathlib import Path
import subprocess, sys, os, hashlib, json
ROOT=Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
print("CPU_ONLY_REVIEW_SOURCE_READY", subprocess.check_output(["git","rev-parse","--short","HEAD"],text=True).strip())


## 2 — Upload the exact ORIGINAL private Stage 13 results ZIP

Only upload the 4-member original `policyproof_native40_dev48_paired_gpu_results.zip`. The source SHA256 is checked **before any output**. Do not upload your original adapter weights here: this is a review of already-saved responses.


In [ ]:
from google.colab import files
BUNDLE=Path("/content/policyproof_native40_dev48_paired_gpu_results.zip")
EXPECTED="e0319c83fbb87ef8570ca0c42caf68bea4a4fef1f8ae320bbe11e9a7c1761c68"
if not BUNDLE.is_file():
    os.chdir("/content")
    uploaded=files.upload()
    assert set(uploaded)=={BUNDLE.name}, "Upload only the original Stage 13 paired GPU results ZIP"
os.chdir(ROOT)
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==EXPECTED, "Source ZIP was modified: STOP"
print("ORIGINAL_NATIVE40_PAIR_BUNDLE_SHA256_VERIFIED")


## 3 — Recompute strict metrics from original raw text and frozen gold

This validates the source ZIP, compares every saved per-case score against a **new calculation from the original raw responses and gold dataset**, and writes only an aggregate JSON. It cannot certify semantic equivalence, passage entailment or GPU execution.


In [ ]:
from scripts.rescore_original_native40_dev48 import rescore_original_bundle
fresh=rescore_original_bundle(ROOT,BUNDLE,EXPECTED)
assert fresh["all_original_saved_case_scores_exactly_reproduced"] is True
assert fresh["new_model_inference_or_training_performed"] is False
assert fresh["raw_private_model_predictions_included"] is False
print("RAW_GOLD_RESCORE_PASSED", fresh["status"])
for name in ("schema_valid","supported_answer_exact","unsupported_abstention_correct",
             "unsupported_claims"):
    v=fresh["metrics"][name]
    print(name, v["raw_before"], "/", v["denominator"], "→",
          v["raw_after"], "/", v["denominator"],
          "improved",v["improved_cases"],"regressed",v["regressed_cases"])
print("NO HUMAN SEMANTIC REVIEW PERFORMED")


## 4 — Optional PRIVATE reviewer packet export (OFF by default)

The packet contains **96 real model outputs in randomized model-blinded A/B slots** (48 paired cases). The gold answer, original model identity, and answerability categories are not included. Reviewers must judge meaning and evidence support themselves.

The separate mapping file is **for the owner only**. **Do not share it with reviewers**, GitHub, HF, or the organizers. The reviewer packet also contains sensitive unpublished model responses and must be shared privately with qualified reviewers only.

If you have at least two independent fluent reviewers who can honestly handle all three languages, change the switch below to `True`. No human votes are generated.


In [ ]:
EXPORT_PRIVATE_BLINDED_REVIEW_PACKET = False  # explicit owner opt-in
if EXPORT_PRIVATE_BLINDED_REVIEW_PACKET:
    from scripts.export_review_from_bundle import export_from_bundle
    import csv
    result=export_from_bundle(
        ROOT, BUNDLE, EXPECTED,
        packet="reviews/dev48_blank.csv",
        key="reviews/dev48_private_mapping.json",
    )
    assert result["response_rows"] == 96
    assert result["human_semantic_review_performed"] is False
    print("PRIVATE_BLINDED_REVIEW_PACKET_CREATED",result["status"])
    print("Download TWO files but keep them separate.")
    print("Share only dev48_blank.csv with reviewers privately.")
    files.download(str(ROOT/"reviews/dev48_blank.csv"))
    print("Keep dev48_private_mapping.json OWNER-ONLY. NEVER share with reviewers.")
    files.download(str(ROOT/"reviews/dev48_private_mapping.json"))
else:
    print("BLIND_REVIEW_EXPORT_OFF: no CSV, identity key or fake human judgments created.")


## 5 — Real reviewers: instructions

Give **each** qualified, independent reviewer a different private copy of the same `dev48_blank.csv` and the [review scoring guidance](https://github.com/betterlifestyle04-commits/PolicyProof-AI/blob/main/docs/NATIVE40_DEV48_HUMAN_REVIEW_HANDOFF.md). They must assign an individual reviewer code and enter **0/1 for every response** in `semantic_correct`, `evidence_supported`, and `unsafe_or_injection_followed`. They must NOT be shown the owner-only mapping, the gold answers, or another reviewer's votes.

Do not edit original evidence columns, renumber cases, invent reviewer answers, or present this as a blind final benchmark. When both reviewers return their filled files, we can run the existing strict `scripts.paired_semantic_review summarize` command, adjudicate genuine disagreements, and report the results as **development-only**.

Never paste human-review answers into this notebook yourself to simulate independent reviewers. An uncompleted review does not block a truthful Track 2a benchmark submission.
